# legal-expand - Demo interactiva completa

**646 siglas legales españolas verificadas** | Fuentes: RAE, BOE, DPEJ...

[![PyPI](https://badge.fury.io/py/legal-expand.svg)](https://pypi.org/project/legal-expand/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.9+](https://img.shields.io/badge/python-3.9+-blue.svg)](https://www.python.org/downloads/)

---

Este notebook demuestra **todas** las funcionalidades del paquete `legal-expand` para Python.

**Repositorio:** https://github.com/686f6c61/pypi-legal-expand

## Indice

1. Instalacion
2. Uso basico
3. Formatos de salida
4. Control global
5. Opciones avanzadas
6. Funciones auxiliares
7. Proteccion de contextos
8. Casos de uso por perfil
9. Ejemplos de documentos reales
10. Extensibilidad
11. Herramientas interactivas

---
## 1. Instalacion

Ejecuta esta celda para instalar el paquete (solo necesario una vez):

In [ ]:
!pip install legal-expand -q

# Verificar instalacion
from legal_expand import __version__
print(f"[OK] legal-expand v{__version__} instalado correctamente")

---
## 2. Uso basico

### 2.1 Expansion simple

In [ ]:
from legal_expand import expandir_siglas

texto = 'La AEAT notifica el IVA'
resultado = expandir_siglas(texto)

print(f"Entrada: {texto}")
print(f"Salida:  {resultado}")

### 2.2 Expansion de multiples siglas

In [ ]:
texto = 'Segun el art. 123 del CC y la LEC, la AEAT debe procesar el BOE.'
resultado = expandir_siglas(texto)

print(f"Entrada:\n{texto}")
print(f"\nSalida:\n{resultado}")

### 2.3 Deteccion de variantes con/sin puntos

In [ ]:
# Con punto
print("Con punto:", expandir_siglas('El art. 5 establece'))

# Sin punto
print("Sin punto:", expandir_siglas('El art 5 establece'))

# Mayusculas
print("Mayusculas:", expandir_siglas('La AEAT notifica'))

---
## 3. Formatos de salida

### 3.1 Formato plain (texto plano, por defecto)

In [ ]:
from legal_expand import expandir_siglas, ExpansionOptions

texto = 'La AEAT gestiona el IVA y el IRPF'

resultado = expandir_siglas(texto)  # format='plain' por defecto
print("Formato PLAIN:")
print(resultado)

### 3.2 Formato HTML (con etiquetas abbr para tooltips)

In [ ]:
resultado_html = expandir_siglas(texto, ExpansionOptions(format='html'))

print("Formato HTML (codigo):")
print(resultado_html)

In [ ]:
# Renderizar el HTML en el notebook
from IPython.display import HTML, display

print("HTML renderizado (pasa el raton sobre las siglas):")
display(HTML(f"<div style='font-size:18px; padding:15px; background:#f0f7ff; border-radius:8px; border:1px solid #cce5ff;'>{resultado_html}</div>"))

### 3.3 Formato structured (objeto con metadatos completos)

In [ ]:
resultado = expandir_siglas('AEAT, IVA y BOE', ExpansionOptions(format='structured'))

print("Formato STRUCTURED:")
print(f"\n  original_text: {resultado.original_text}")
print(f"  expanded_text: {resultado.expanded_text}")
print(f"\n  Estadisticas:")
print(f"     - Total encontradas: {resultado.stats.total_acronyms_found}")
print(f"     - Total expandidas: {resultado.stats.total_expanded}")
print(f"     - Ambiguas no expandidas: {resultado.stats.ambiguous_not_expanded}")
print(f"\n  Siglas encontradas:")
for acronym in resultado.acronyms:
    print(f"     * {acronym.acronym} -> {acronym.expansion}")
    print(f"       Posicion: [{acronym.position.start}:{acronym.position.end}]")

---
## 4. Control global

### 4.1 Configuracion global por defecto

In [ ]:
from legal_expand import (
    configurar_globalmente,
    obtener_configuracion_global,
    resetear_configuracion,
    GlobalConfig,
    ExpansionOptions
)

# Ver configuracion actual
config = obtener_configuracion_global()
print("Configuracion actual:")
print(f"   enabled: {config.enabled}")
print(f"   format: {config.default_options.format}")

### 4.2 Desactivar/activar expansion globalmente

In [ ]:
# Desactivar globalmente
configurar_globalmente(GlobalConfig(enabled=False))
print("[X] Expansion DESACTIVADA globalmente:")
print(f"   {expandir_siglas('La AEAT notifica el IVA')}")

# Reactivar
resetear_configuracion()
print("\n[OK] Expansion ACTIVADA globalmente:")
print(f"   {expandir_siglas('La AEAT notifica el IVA')}")

### 4.3 Override con force_expansion

In [ ]:
# Desactivar globalmente
configurar_globalmente(GlobalConfig(enabled=False))

# Pero forzar expansion en esta llamada especifica
print("Con force_expansion=True (ignora config global):")
resultado = expandir_siglas('La AEAT notifica', ExpansionOptions(force_expansion=True))
print(f"   {resultado}")

# Tambien funciona al reves
resetear_configuracion()  # Activar globalmente
print("\nCon force_expansion=False (ignora config global):")
resultado = expandir_siglas('La AEAT notifica', ExpansionOptions(force_expansion=False))
print(f"   {resultado}")

resetear_configuracion()

### 4.4 Configurar opciones por defecto

In [ ]:
# Configurar HTML como formato por defecto para toda la app
configurar_globalmente(GlobalConfig(
    enabled=True,
    default_options=ExpansionOptions(
        format='html',
        expand_only_first=True
    )
))

print("Configuracion cambiada a HTML + expand_only_first")
resultado = expandir_siglas('AEAT procesa. AEAT notifica.')
print(f"   {resultado}")

# Restaurar
resetear_configuracion()
print("\n[OK] Configuracion restaurada a valores por defecto")

---
## 5. Opciones avanzadas

### 5.1 expand_only_first (solo primera ocurrencia)

**Ideal para documentos largos** como sentencias, contratos o informes:

In [ ]:
documento = """La AEAT ha notificado la liquidacion.
La AEAT requiere documentacion adicional.
El contribuyente debe presentar ante la AEAT los justificantes.
La AEAT verificara los datos presentados."""

print("SIN expand_only_first (todas se expanden):")
print(expandir_siglas(documento))

print("\n" + "="*70 + "\n")

print("CON expand_only_first=True (solo la primera):")
print(expandir_siglas(documento, ExpansionOptions(expand_only_first=True)))

### 5.2 exclude (excluir siglas especificas)

In [ ]:
texto = 'AEAT, BOE, CC, IVA y IRPF'

# Excluir CC e IVA
resultado = expandir_siglas(texto, ExpansionOptions(exclude=['CC', 'IVA']))

print(f"Entrada: {texto}")
print(f"Excluyendo: CC, IVA")
print(f"Salida: {resultado}")

### 5.3 include (incluir solo siglas especificas)

In [ ]:
texto = 'AEAT, BOE, CC, IVA y IRPF'

# Solo expandir AEAT y BOE
resultado = expandir_siglas(texto, ExpansionOptions(include=['AEAT', 'BOE']))

print(f"Entrada: {texto}")
print(f"Solo incluyendo: AEAT, BOE")
print(f"Salida: {resultado}")

### 5.4 Combinar multiples opciones

In [ ]:
texto = 'La AEAT gestiona el IVA. La AEAT tambien cobra el IRPF y supervisa el BOE.'

resultado = expandir_siglas(texto, ExpansionOptions(
    format='html',
    expand_only_first=True,
    exclude=['BOE']
))

print("Opciones combinadas: HTML + expand_only_first + exclude=['BOE']")
print(f"\nResultado:")
display(HTML(f"<div style='padding:10px; background:#f5f5f5; border-radius:5px;'>{resultado}</div>"))

---
## 6. Funciones auxiliares

### 6.1 buscar_sigla() - Informacion de una sigla

In [ ]:
from legal_expand import buscar_sigla

# Buscar AEAT
info = buscar_sigla('AEAT')

print("buscar_sigla('AEAT'):")
print(f"   Sigla: {info.acronym}")
print(f"   Significados: {info.meanings}")
print(f"   Tiene duplicados: {info.has_duplicates}")

# Buscar una sigla inexistente
info2 = buscar_sigla('XYZ')
print(f"\nbuscar_sigla('XYZ'): {info2}")

### 6.2 listar_siglas() - Todas las siglas disponibles

In [ ]:
from legal_expand import listar_siglas

siglas = listar_siglas()

print(f"Total de siglas disponibles: {len(siglas)}")
print(f"\nPrimeras 30 siglas:")
print(siglas[:30])
print(f"\nUltimas 30 siglas:")
print(siglas[-30:])

### 6.3 obtener_estadisticas() - Metricas del diccionario

In [ ]:
from legal_expand import obtener_estadisticas

stats = obtener_estadisticas()

print("Estadisticas del diccionario:")
print(f"   * Total de siglas: {stats.total_acronyms}")
print(f"   * Siglas con multiples significados: {stats.acronyms_with_duplicates}")
print(f"   * Siglas con puntuacion (art., num., etc.): {stats.acronyms_with_punctuation}")

---
## 7. Proteccion de contextos

El paquete **no expande** siglas que aparecen en URL, correos electronicos o codigo:

In [ ]:
# URL: la sigla en la URL no se expande
texto = 'Visita https://aeat.es para mas informacion sobre AEAT'
print(f"URL:\n   {expandir_siglas(texto)}")

# Correo electronico: la sigla en el email no se expande
texto = 'Contacta con info@aeat.es o con la AEAT directamente'
print(f"\nCorreo electronico:\n   {expandir_siglas(texto)}")

# Codigo: las siglas dentro de bloques de codigo no se expanden
texto = '''Texto normal con AEAT.
```python
AEAT = 'test'
```
Mas texto con AEAT.'''
print(f"\nCodigo markdown:\n{expandir_siglas(texto)}")

---
## 8. Casos de uso por perfil

### 8.1 Estudiantes de derecho

In [ ]:
# Estudiantes: hacer los apuntes mas comprensibles
apuntes = 'El TS aplica el art. 24 CE en materia de tutela judicial efectiva.'
apuntes_expandidos = expandir_siglas(apuntes)

print("ESTUDIANTES DE DERECHO")
print(f"\nApuntes originales:\n   {apuntes}")
print(f"\nApuntes expandidos:\n   {apuntes_expandidos}")

### 8.2 Opositores

In [ ]:
# Opositores: documentos largos con expand_only_first
temario = '''La AEAT gestiona el IVA e IRPF segun la LGT.
La AEAT tambien supervisa el cumplimiento de la LIRPF.
El art. 123 de la LGT establece los plazos de prescripcion.
La AEAT puede iniciar procedimientos de inspeccion.'''

resultado = expandir_siglas(temario, ExpansionOptions(expand_only_first=True))

print("OPOSITORES")
print(f"\nTemario expandido (solo primera ocurrencia):")
print(resultado)

### 8.3 Academias y centros de formacion

In [ ]:
# Academias: formato HTML para plataformas web
material = 'La AEAT gestiona el IVA segun la LGT y el RGAT.'
material_html = expandir_siglas(material, ExpansionOptions(format='html'))

print("ACADEMIAS Y CENTROS DE FORMACION")
print("\nMaterial formateado para web:")
display(HTML(f"<div style='font-size:16px; padding:15px; background:#e8f5e9; border-radius:8px;'>{material_html}</div>"))

### 8.4 Despachos de abogados

In [ ]:
# Despachos: documentos para clientes sin conocimientos tecnicos
contrato = '''CONTRATO DE PRESTACION DE SERVICIOS

De conformidad con el CC y la LAU, el Sr. Garcia contrata
los servicios profesionales. El IVA aplicable sera del 21%
segun la LIVA. En caso de controversia, las partes se someten
a la jurisdiccion de los Juzgados de Madrid conforme a la LEC.'''

contrato_claro = expandir_siglas(contrato, ExpansionOptions(
    expand_only_first=True,
    exclude=['Sr.']  # Excluir abreviaturas comunes
))

print("DESPACHOS DE ABOGADOS")
print("\nContrato adaptado para cliente:")
print(contrato_claro)

### 8.5 Administracion publica

In [ ]:
# Administracion: comunicacion accesible para ciudadanos
comunicado = '''COMUNICADO OFICIAL

La AEAT informa que el plazo para la declaracion del IRPF
finaliza el 30 de junio. Los contribuyentes pueden consultar
el BOE para mas informacion sobre deducciones aplicables.'''

comunicado_ciudadano = expandir_siglas(comunicado, ExpansionOptions(format='html'))

print("ADMINISTRACION PUBLICA")
print("\nComunicado accesible para ciudadanos:")
display(HTML(f"<div style='font-size:16px; padding:15px; background:#fff3e0; border-radius:8px;'>{comunicado_ciudadano}</div>"))

### 8.6 Desarrolladores (integracion con LLM)

In [ ]:
# Desarrolladores: preprocesar texto antes de enviar a un LLM
documento_legal = 'La AEAT notifica la liquidacion del IVA conforme a la LGT.'

# 1. Expandir siglas
documento_expandido = expandir_siglas(documento_legal)

# 2. El texto expandido esta listo para enviar al LLM
print("DESARROLLADORES - Integracion con LLM")
print(f"\nDocumento original:\n   {documento_legal}")
print(f"\nDocumento preprocesado para LLM:\n   {documento_expandido}")
print("\nBeneficios:")
print("   * El LLM entiende el contexto completo")
print("   * Mayor precision en las respuestas")
print("   * Consistencia en el procesamiento")

---
## 9. Ejemplos de documentos reales

### 9.1 Sentencia judicial

In [ ]:
sentencia = '''Fundamentos de Derecho: El TS, en su sentencia de fecha 15 de marzo de 2024,
confirma la doctrina establecida por el TC respecto a la aplicacion del art. 24 CE
en materia de tutela judicial efectiva. La AEAT, como parte demandada, alego la
prescripcion del derecho conforme al art. 66 de la LGT. El tribunal considera que
la liquidacion del IVA e IRPF se ajusta a derecho, conforme a los arts. 108 y 123
de la LRJSP. La defensa invoco la vulneracion del principio de proporcionalidad
consagrado en la CE, pero el TS desestimo el recurso. Se condena en costas a la
parte recurrente segun el art. 394 LEC.'''

resultado = expandir_siglas(sentencia, ExpansionOptions(expand_only_first=True))

print("SENTENCIA JUDICIAL")
print("="*70)
print(resultado)

### 9.2 Contrato de arrendamiento

In [ ]:
contrato = '''CONTRATO DE ARRENDAMIENTO. En Madrid, a 10 de enero de 2024.
CLAUSULA PRIMERA: El arrendador cede el uso de la vivienda conforme a la LAU.
CLAUSULA SEGUNDA: La renta mensual se abonara mediante transferencia bancaria.
CLAUSULA TERCERA: El arrendatario se compromete al pago del IBI y gastos de comunidad.
CLAUSULA CUARTA: Conforme al CC, el incumplimiento de las obligaciones facultara
al arrendador para resolver el contrato.
CLAUSULA QUINTA: Las partes se someten a la jurisdiccion de los Juzgados de Madrid
conforme a la LEC.
CLAUSULA SEXTA: De conformidad con la LOPD, el arrendador se compromete a la
proteccion de datos personales del arrendatario.'''

resultado = expandir_siglas(contrato, ExpansionOptions(expand_only_first=True))

print("CONTRATO DE ARRENDAMIENTO")
print("="*70)
print(resultado)

### 9.3 Notificacion de Hacienda

In [ ]:
notificacion = '''NOTIFICACION DE LIQUIDACION PROVISIONAL. Expediente: 2024/123456.
La AEAT, en uso de las facultades conferidas por la LGT, procede a notificar la
liquidacion provisional del IVA correspondiente al ejercicio fiscal 2023 por
importe de 15.000 euros. Asimismo, se liquida el IRPF con una cuota de 8.500 euros.
La presente liquidacion se practica conforme a los arts. 101 y 102 de la LGT.
De acuerdo con el art. 123 de la LRJSP, el contribuyente dispone de un plazo de
30 dias habiles para presentar alegaciones.'''

resultado = expandir_siglas(notificacion, ExpansionOptions(expand_only_first=True))

print("NOTIFICACION DE LIQUIDACION")
print("="*70)
print(resultado)

### 9.4 Texto academico

In [ ]:
academico = '''ANALISIS JURISPRUDENCIAL DE LOS DERECHOS FUNDAMENTALES.
Segun la doctrina consolidada del TS y la jurisprudencia del TEDH, los derechos
fundamentales consagrados en la CE deben interpretarse de conformidad con los
tratados internacionales, especialmente la DUDH y el CEDH. El TC ha reiterado
en multiples sentencias la primacia del art. 24 CE sobre el derecho a la tutela
judicial efectiva. En este sentido, la STC 120/1990 establecio que el derecho
reconocido en el art. 24 CE comprende el acceso a los recursos.'''

resultado = expandir_siglas(academico, ExpansionOptions(expand_only_first=True))

print("TEXTO ACADEMICO")
print("="*70)
print(resultado)

### 9.5 Real decreto (BOE)

In [ ]:
boe = '''REAL DECRETO 123/2024, de 15 de marzo, publicado en el BOE num. 65,
por el que se modifica el RD 1065/2007 sobre la aplicacion del IVA en
operaciones inmobiliarias.
PREAMBULO: La presente norma tiene por objeto adaptar la normativa fiscal
a las directivas de la UE en materia tributaria.
ARTICULO PRIMERO: Se modifica el art. 20 del RD 1065/2007 en relacion con
las exenciones del IVA.
ARTICULO SEGUNDO: La AEAT debera adaptar sus procedimientos de gestion
conforme a lo establecido en la LGT y en la LRJSP.
DISPOSICION FINAL: El presente RD entrara en vigor el dia siguiente a su
publicacion en el BOE.'''

resultado = expandir_siglas(boe, ExpansionOptions(expand_only_first=True))

print("REAL DECRETO (BOE)")
print("="*70)
print(resultado)

### 9.6 Resolucion administrativa (AEPD)

In [ ]:
resolucion = '''RESOLUCION DEL PROCEDIMIENTO SANCIONADOR. Expediente: SAN/2024/789.
Visto el expediente instruido por la AEPD en relacion con la presunta infraccion
de la LOPD y del RGPD por parte de la empresa ACME S.L., se dicta la presente resolucion.
ANTECEDENTES:
Primero. Con fecha 10 de enero de 2024, tuvo entrada en la AEPD reclamacion contra
ACME S.L. por vulneracion del art. 6 del RGPD.
Segundo. La AEPD inicio el procedimiento sancionador conforme al art. 63 de la LOPD.
FUNDAMENTOS DE DERECHO:
Primero. La AEPD es competente conforme al art. 58 del RGPD y arts. 47 y 68 de la LOPD.
Segundo. Se considera acreditada la infraccion del art. 6.1 del RGPD.
RESUELVO: Imponer a ACME S.L. una multa de 10.000 euros conforme al art. 83 del RGPD
y art. 74 de la LOPD.'''

resultado = expandir_siglas(resolucion, ExpansionOptions(expand_only_first=True))

print("RESOLUCION ADMINISTRATIVA")
print("="*70)
print(resultado)

---
## 10. Extensibilidad

### 10.1 Crear formatter personalizado

In [ ]:
from legal_expand import FormatterFactory, Formatter
from legal_expand.types import MatchInfo

class MarkdownFormatter(Formatter):
    """Formatter que genera Markdown con negrita y cursiva."""
    def format(self, original_text: str, matches: list[MatchInfo]) -> str:
        if not matches:
            return original_text

        sorted_matches = sorted(matches, key=lambda m: m.start_pos, reverse=True)
        result = original_text

        for match in sorted_matches:
            acronym = original_text[match.start_pos:match.end_pos]
            replacement = f"**{acronym}** (*{match.expansion}*)"
            result = result[:match.start_pos] + replacement + result[match.end_pos:]

        return result

# Registrar el formatter personalizado
FormatterFactory.register_formatter('markdown', MarkdownFormatter())

# Listar formatters disponibles
print("Formatters disponibles:", FormatterFactory.list_formatters())

### 10.2 Generador de glosario automatico

In [ ]:
def generar_documento_con_glosario(texto: str) -> str:
    """Genera un documento expandido con glosario automatico al final."""
    resultado = expandir_siglas(texto, ExpansionOptions(
        format='structured',
        expand_only_first=True
    ))

    # Generar glosario unico (sin duplicados)
    glosario = {}
    for sigla in resultado.acronyms:
        if sigla.acronym not in glosario:
            glosario[sigla.acronym] = sigla.expansion

    # Construir documento final
    output = resultado.expanded_text

    if glosario:
        output += '\n\n' + '='*50
        output += '\nGLOSARIO DE SIGLAS\n'
        output += '='*50 + '\n\n'
        for sigla, significado in sorted(glosario.items()):
            output += f"* {sigla}: {significado}\n"

    return output

# Ejemplo
documento = '''La AEAT gestiona el IVA y el IRPF segun normativa del BOE.
El TS y el TC interpretan la CE en materia de derechos fundamentales.
La LOPD protege los datos personales conforme al RGPD.'''

print(generar_documento_con_glosario(documento))

---
## 11. Herramientas interactivas

### 11.1 Buscador de siglas

In [ ]:
# @title Buscar una sigla { run: "auto" }
sigla_a_buscar = "AEAT" # @param {type:"string"}

info = buscar_sigla(sigla_a_buscar.upper())

if info:
    print(f"[OK] Sigla encontrada: {info.acronym}")
    print(f"\nSignificado(s):")
    for meaning in info.meanings:
        print(f"   * {meaning}")
    if info.has_duplicates:
        print(f"\n[!] Esta sigla tiene multiples significados")
else:
    print(f"[X] Sigla '{sigla_a_buscar}' no encontrada en el diccionario")
    print(f"\nSugerencia: prueba con siglas como AEAT, IVA, BOE, TS, TC, CC, LEC...")

### 11.2 Expansor de texto

In [ ]:
# @title Expansor de texto { run: "auto" }
mi_texto = "La AEAT gestiona el IVA y el IRPF" # @param {type:"string"}
formato = "plain" # @param ["plain", "html", "structured", "markdown"]
solo_primera = False # @param {type:"boolean"}

opciones = ExpansionOptions(
    format=formato,
    expand_only_first=solo_primera
)

resultado = expandir_siglas(mi_texto, opciones)

print(f"Entrada: {mi_texto}")
print(f"Formato: {formato} | Solo primera: {solo_primera}")
print(f"\nResultado:")

if formato == 'structured':
    print(f"   Expandido: {resultado.expanded_text}")
    print(f"   Siglas encontradas: {resultado.stats.total_acronyms_found}")
    for a in resultado.acronyms:
        print(f"   * {a.acronym} -> {a.expansion}")
elif formato == 'html':
    display(HTML(f"<div style='padding:10px; background:#f5f5f5; border-radius:5px;'>{resultado}</div>"))
else:
    print(f"   {resultado}")

# @title Expansor de texto { run: "auto" }
mi_texto = "La AEAT gestiona el IVA y el IRPF" # @param {type:"string"}
formato = "plain" # @param ["plain", "html", "structured"]
solo_primera = False # @param {type:"boolean"}

opciones = ExpansionOptions(
    format=formato,
    expand_only_first=solo_primera
)

resultado = expandir_siglas(mi_texto, opciones)

print(f"Entrada: {mi_texto}")
print(f"Formato: {formato} | Solo primera: {solo_primera}")
print(f"\nResultado:")

if formato == 'structured':
    print(f"   Expandido: {resultado.expanded_text}")
    print(f"   Siglas encontradas: {resultado.stats.total_acronyms_found}")
    for a in resultado.acronyms:
        print(f"   * {a.acronym} -> {a.expansion}")
elif formato == 'html':
    display(HTML(f"<div style='padding:10px; background:#f5f5f5; border-radius:5px;'>{resultado}</div>"))
else:
    print(f"   {resultado}")

In [ ]:
# @title Analizador de documento

documento_analizar = '''La AEAT ha emitido una nueva instruccion sobre el IVA
y el IRPF. Segun el art. 123 de la LGT, los contribuyentes
deben presentar sus declaraciones conforme al BOE.
El TS y el TC han confirmado esta interpretacion.''' # @param {type:"string"}

resultado = expandir_siglas(documento_analizar, ExpansionOptions(format='structured'))

print("ANALISIS DEL DOCUMENTO")
print("="*50)
print(f"\nTexto original:\n{documento_analizar}")
print(f"\nEstadisticas:")
print(f"   * Siglas encontradas: {resultado.stats.total_acronyms_found}")
print(f"   * Siglas expandidas: {resultado.stats.total_expanded}")
print(f"\nSiglas detectadas:")
for acronym in resultado.acronyms:
    print(f"   * {acronym.acronym}: {acronym.expansion}")
    print(f"     Posicion: caracteres {acronym.position.start}-{acronym.position.end}")
print(f"\nTexto expandido:\n{resultado.expanded_text}")

---

## Recursos

| Recurso | URL |
|---------|-----|
| **PyPI** | https://pypi.org/project/legal-expand/ |
| **GitHub** | https://github.com/686f6c61/pypi-legal-expand |
| **npm (version JavaScript)** | https://www.npmjs.com/package/legal-expand |

### Fuentes de las siglas

- **RAE** - Real Academia Espanola (Libro de Estilo de la Justicia)
- **DPEJ** - Diccionario Panhispanico del Espanol Juridico
- **BOE** - Boletin Oficial del Estado

---

**Licencia:** MIT | **Autor:** [@686f6c61](https://github.com/686f6c61)